# Fispact Input

f4enix deals with fispact inputs and outputs through the `pypact` package. A light-weight wrapper of `pypact.InputData` class is available in f4enix which allows to more easily integrate it with f4enix `Irradiation` and `Material` classes.

The description of `Irradiation` and `Material` classes are out of scope of this section.

In [2]:
from f4enix.input.materials import Material
from f4enix.input.libmanager import LibManager
from f4enix.core.irradiation import IrradiationScenario, Pulse

material = Material.from_zaids(
    [('Fe', 0.5), ('C', 0.5)],
    LibManager(),
    lib="31c",
)

pulse_1 = Pulse(time=10, intensity=1.0)
pulse_2 = Pulse(time=20, intensity=2.0)
cooling = Pulse(time=30, intensity=0.0)

irrad_scenario = IrradiationScenario(
    pulses=[pulse_1, pulse_2],
    cooling_times=[cooling],
)

f4enix `FispactInp` object has only one attribute, which is the `pypact.InputData` that it wraps.

In [3]:
from f4enix.input.fispact_inp import FispactInp

fisp = FispactInp()
fisp.inp

Convenience methods are defined on the wrapper to help insert materials and irradiation scenarios
using native f4enix objects.

In [4]:
from tempfile import TemporaryDirectory
from pathlib import Path

fisp.add_material(
    material,
    style='mass', # this can be either 'mass' or 'fuel' following fispact keywords
    mass=1.0 # in kg
    )
fisp.add_irradiation_scenario(irrad_scenario)

# Use all pypact methods to finish off the input if needed
fisp.inp.enableHalflifeInOutput(True)
fisp.inp.enableHazardsInOutput()
fisp.inp.setAtomsThreshold(100000)  # MIND card
# ...

with TemporaryDirectory() as tmpdir:
    outpath = Path(f"{tmpdir}/fispact_input.inp")
    fisp.save(outpath)  # save the file to text

    # read file contents
    print(outpath.read_text())

<< CONTROL PHASE >>
<< the minimum cross section (barns) for inclusion in pathways analysis >>
XSTHRESHOLD 1e-12
<< end control >>
FISPACT 
* run

<< INITIALIZATION PHASE >>
<< output half life values >>
HALF 
<< output ingestion and inhalation values >>
HAZARDS 
<< set the target via MASS >>
MASS 1.0 2
Fe 8.2299760518E-01
C 1.7700239482E-01 
<< set the threshold for atoms in the inventory >>
MIND 100000

<< INVENTORY PHASE >>
<< irradiation schedule >>
FLUX 1.0000000000E+00
TIME 1.0000000000E+01 SECS
ATOMS 
FLUX 2.0000000000E+00
TIME 2.0000000000E+01 SECS
ATOMS 
<< end of irradiation >>
FLUX 0.0
ZERO 
TIME 3.0000000000E+01 SECS
ATOMS 
<< end of cooling >>

<< end of input >>
END 
* end

